In [2]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [4]:
from rag_helper import RAGBase


instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [5]:
answer = assistant.rag('How do I run Ollama locally?')
print(answer)

To run Ollama locally:

1. Install Ollama from https://ollama.com/download for your OS:
   - macOS: download and install the `.pkg`
   - Windows: download and install the `.msi`
   - Linux: run
     ```bash
     curl -fsSL https://ollama.com/install.sh | sh
     ```

2. Open a terminal and start a model:
   ```bash
   ollama run llama3
   ```

This will download the LLaMA 3 model, start it locally, and open a chat-like interface.

To test that the local server is running, you can use:
```bash
curl http://localhost:11434
```

If you get a connection refused error, restart the Ollama server with:
```bash
!nohup ollama serve > nohup.out 2>&1 &
```


In [ ]:
def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [9]:
messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
)

response.output_text

'Maybe — it depends on the course’s enrollment rules and whether it’s still open.\n\nIf you want, I can help you figure it out. Usually you’d check:\n- whether the course is still accepting students,\n- if there’s a registration deadline,\n- whether you need an invitation, approval, or prerequisites,\n- and whether there’s a waitlist.\n\nIf you tell me the course name or share the course page/info, I can help you determine if you can still join.'

In [7]:
search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

In [10]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course after it has started? enrollment late join discovered course"}', call_id='call_dHzPPrgVJh3DzPSPFAjcBnTJ', name='search', type='function_call', id='fc_07003e04de0c4355006a457fbb8844819c8655948f650defc8', namespace=None, status='completed')]

In [11]:
import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

In [12]:
messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

In [13]:
response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output_text

'Yes, you can join it anytime and start learning. The materials are available, and you can follow along at your own pace.\n\nIf you want a certificate, though, you need to submit your project while the course is still accepting submissions.'

In [14]:
def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))

Total cost: $ 0.0001176


In [15]:
def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [16]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'


messages = [
    {'role': 'developer', 'content': instructions},
    {'role': 'user', 'content': question}
]

In [17]:
response = openai_client.responses.create(
    model='gpt-5.4-mini',
    input=messages,
    tools=[search_tool]
)

In [18]:
messages.extend(response.output)

for item in response.output:
    if item.type == 'function_call':
        print('function_call:', item.name, item.arguments)
        call_output = make_call(item)
        messages.append(call_output)

    elif item.type == 'message':
        print('ASSISTANT:')
        print(item.content[0].text)

function_call: search {"query":"join course discovered course can I join enrollment registration late join"}
function_call: search {"query":"course FAQ enrollment join after start discovered the course can I join"}


In [19]:
def agent_loop(instructions, question, model='gpt-5.4-mini') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [20]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

At the end, ask if there are other areas that the user wants to explore.
"""

question = 'I just discovered the course. Can I join it?'
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"join course late enrollment discovered course can I join it"}
iteration #2...
ASSISTANT:
Yes — you can still join the course.

If you want to receive a certificate, make sure you submit your project while the course is still accepting submissions.

If you want, I can also help you with how to start the course or explain the certificate requirements.


In [21]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searchers. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
"""

question = "what's queen gambit?"

result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"queen gambit"}
iteration #2...
function_call: search {"query":"queen's gambit chess opening"}
iteration #3...
function_call: search {"query":"queen gambit queen's gambit chess opening FAQ"}
iteration #4...
ASSISTANT:
I couldn’t find anything in the course FAQ about “queen gambit,” so it may be off-topic or not covered here.

If you meant **“Queen’s Gambit”** as a chess opening, I’m not able to explain it unless it’s in the course FAQ. If you want, I can help look for any course-related topic instead—are there other areas you want to explore?
